# Claude API のトークン数と料金を事前に見積もる

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/13_llm_api/claude_count_tokens.ipynb)

NLP クックブック 13-10 ／ LLM の API (Gemini と Claude)

> API キーが必要 (Colab 左の 🔑 シークレットに登録)

データセット全体を API に投げる前に、何トークンになり、いくらかかるのかを見積もりたい。

In [ ]:
!pip install -q anthropic

In [ ]:
import anthropic
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))
MODEL = "claude-opus-5-5"
SYSTEM = "次の文の文法誤りを訂正し、訂正後の文だけを出力してください。"

def count(text):
    return client.messages.count_tokens(model=MODEL, system=SYSTEM, messages=[{"role": "user", "content": text}]).input_tokens

for text in ["Natural language processing is fun.", "自然言語処理は楽しい。", "吾輩は猫である。名前はまだ無い。どこで生れたかとんと見当がつかぬ。"]:
    print(count(text), "トークン |", len(text), "文字 |", text)

In [ ]:
# データセット全体の見積もり: 一部をサンプリングして平均をとり、件数を掛ける
import random

dataset = ["私は学校を行きます。", "昨日は雨が降ります。", "この本は面白いでした。"] * 1000   # 自分のデータに置き換える
sample = random.Random(0).sample(dataset, 20)
avg_in = sum(count(t) for t in sample) / len(sample)
avg_out = 40                                   # 出力トークン数は、少数を実際に処理して実測する
PRICE_IN, PRICE_OUT = 4.00, 20.00              # 100 万トークンあたりのドル。最新の料金表で確認すること

cost = len(dataset) * (avg_in * PRICE_IN + avg_out * PRICE_OUT) / 1e6
print(f"{len(dataset)} 件 × 入力 {avg_in:.0f} + 出力 {avg_out} トークン ≒ ${cost:.2f} (Batches API なら ${cost / 2:.2f})")

- トークン数はモデルによって違う。必ず実際に使うモデルの ID で数える。他社のトークナイザ (tiktoken など) での推定は当てにならない。
- 出力側に思考 (thinking) のトークンも含まれて課金される点に注意。事前に数十件で実測するのが確実。
- トークン数のカウント自体は無料 (レート制限はある)。